In [1]:
from roboticstoolbox import *
from spatialmath import *
from math import pi
import numpy as np
import time
import math
import gripper
import DrEmpower_can as dr
import arm_robot as robot
import sys
sys.path.append('/daran')

In [2]:
%matplotlib tk

标准DH


In [3]:
DFbot = DHRobot(
    [
                    RevoluteDH(alpha=-np.pi/2,qlim=np.array([-np.pi,np.pi])),
                    RevoluteDH(a=-0.15,qlim=np.array([-np.pi,np.pi])),
                    RevoluteDH(a=-0.15,qlim=np.array([-np.pi,np.pi])),
                    RevoluteDH(d=-0.058,alpha=-np.pi/2,offset=np.pi/2,qlim=np.array([-np.pi,np.pi])),
                    RevoluteDH(d=0.08,alpha=np.pi/2,qlim=np.array([-np.pi,3/2*np.pi])),
                    RevoluteDH(d=0.042,qlim=np.array([-np.pi,3/2*np.pi])),
                  
    ],
    name="DFbot",
)

In [4]:
DFbot

DHRobot: DFbot, 6 joints (RRRRRR), dynamics, standard DH parameters
┌───────────┬────────┬───────┬────────┬─────────┬────────┐
│    θⱼ     │   dⱼ   │  aⱼ   │   ⍺ⱼ   │   q⁻    │   q⁺   │
├───────────┼────────┼───────┼────────┼─────────┼────────┤
│  q1       │      0 │     0 │ -90.0° │ -180.0° │ 180.0° │
│  q2       │      0 │ -0.15 │   0.0° │ -180.0° │ 180.0° │
│  q3       │      0 │ -0.15 │   0.0° │ -180.0° │ 180.0° │
│  q4 + 90° │ -0.058 │     0 │ -90.0° │ -180.0° │ 180.0° │
│  q5       │   0.08 │     0 │  90.0° │ -180.0° │ 270.0° │
│  q6       │  0.042 │     0 │   0.0° │ -180.0° │ 270.0° │
└───────────┴────────┴───────┴────────┴─────────┴────────┘

┌──┬──┐
└──┴──┘

In [ ]:
DFbot.plot([0,np.pi/3,-np.pi/3,0,np.pi,0])

PyPlot3D backend, t = 0.05, scene:
  robot: Text(0.0, 0.0, 'DFbot')

In [23]:
DFbot.plot([0,np.pi/2,0,0,0,0])

PyPlot3D backend, t = 0.05, scene:
  robot: Text(0.0, 0.0, 'DFbot')

In [4]:

l_p = 0 # 工具参考点到电机输出轴表面的距离，单位mm（所有尺寸参数皆为mm）
l_p_mass_center = 0 # 工具（负载）质心到 6 号关节输出面的距离
G_p = 0 # 负载重量，单位kg，所有重量单位皆为kg
max_list_temp = [180, 215, 149, 142, 179, 179]  # 关节模型角度最大值,1号关节目的是保护线缆，并且到达工作空间边缘；2号关节到达工作空间边缘；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达
min_list_temp = [-90, -15, -149, -142, -179, -179]  # 关节模型角度最小值，1号关节目的是不装到装在桌边的竖杆（安装摄像头）；2号关节目的是在伸直的时候不打到桌子；3、4号关节目的是不产生杆件干涉；5号关节因有滑环，不考虑；6号关节保证工作空间内全部到达

# # 机械臂对象初始化函数函数
ro = robot.arm_robot(L_p=l_p, L_p_mass_center=l_p_mass_center, MAX_list_temp=max_list_temp, MIN_list_temp=min_list_temp, G_p=G_p)

位置增益 P：10.0
积分增益 I：5.0
转速增益 D：0.550000011920929
机械臂第  1  号关节的 PID 修改为： [10.0, 5.0, 0.550000011920929]
位置增益 P：10.5600004196167
积分增益 I：4.949999809265137
转速增益 D：0.39100000262260437
机械臂第  2  号关节的 PID 修改为： [10.5600004196167, 4.949999809265137, 0.39100000262260437]
位置增益 P：10.5600004196167
积分增益 I：4.949999809265137
转速增益 D：0.39100000262260437
机械臂第  3  号关节的 PID 修改为： [10.5600004196167, 4.949999809265137, 0.39100000262260437]
位置增益 P：10.0
积分增益 I：9.0
转速增益 D：0.5
机械臂第  4  号关节的 PID 修改为： [10.0, 9.0, 0.5]
位置增益 P：12.0
积分增益 I：5.0
转速增益 D：0.10000000149011612
机械臂第  5  号关节的 PID 修改为： [12.0, 5.0, 0.10000000149011612]
位置增益 P：12.0
积分增益 I：5.0
转速增益 D：0.09600000083446503
机械臂第  6  号关节的 PID 修改为： [12.0, 5.0, 0.09600000083446503]
初始化成功


驱动机械臂到垂直状态

In [5]:
joint_angle = [0,90,0,0,0,0]
ro.set_arm_joints(joint_angle)

True

读取机械臂当前的模型角度，返加6个角度值，单位为角度

In [11]:
joint_angle = ro.detect_joints()

关节 1 模型角度为: -0.0
关节 2 模型角度为: 100.0
关节 3 模型角度为: -86.8
关节 4 模型角度为: -13.2
关节 5 模型角度为: 90.0
关节 6 模型角度为: -0.0


将关节角度转换为弧度

In [7]:
joint_radian = np.array(joint_angle) * np.pi / 180.0

显示机器人模型

In [8]:
DFbot.plot(joint_radian)

PyPlot3D backend, t = 0.05, scene:
  robot: Text(0.0, 0.0, 'DFbot')

机器人关节改变一定的角度

In [6]:

ro.detect_pose()

当前机械臂末端x坐标为: 199.9896087809345; y坐标为： 58.0; z坐标为： 139.97379346842956; Pitch角为: 6.3611093629270335e-15; Roll角为: 90.0; Yaw角为: -0.0


In [6]:
ro.set_arm_pose(pl_temp=[200,58, 140], theta_P_R_Y=[0, 90, 0], speed=2, param=10, mode=1)

In [9]:
####轨迹跟踪函数***********************************************
############画正方形#################
def draw_rectangle(pl=[200, 58, 140], l=30, h=30):
    ''''在水平面上画正方形
    pl: 长方形左上角坐标（起始点），其中pl[2]代表作图平面与全局坐标系z轴的焦点的z坐标
    l: 宽度
    h: 高度
    '''
    n= 50 # 每条边分割的点数（数量越多画得越慢）
    l_delta = l/n
    h_delta = h/n
    pl_list = []
    pl_list.append(pl)
    l1 = pl[1]
    for i in range(1, n+1):
        pl_temp = [pl[0], pl[1]-i*l_delta, pl[2]]
        pl_list.append(pl_temp)
    print(pl_temp)
    for i in range(1, n+1):
        pl_temp1 = [pl_temp[0]-i*h_delta, pl_temp[1], pl_temp[2]]
        pl_list.append(pl_temp1)
    print(pl_temp1)
    for i in range(1, n+1):
        pl_temp2 = [pl_temp1[0], pl_temp1[1]+i*l_delta, pl_temp1[2]]
        pl_list.append(pl_temp2)
    print(pl_temp2)
    for i in range(1, n+1):
        pl_temp3 = [pl_temp2[0]+i*h_delta, pl_temp2[1], pl_temp2[2]]
        pl_list.append(pl_temp3)
    print(pl_temp3)
    print(pl_list)
    return pl_list
#
#
pl_list = draw_rectangle(pl=[200, 58, 140], l=50, h=50) #

pl_list

[200, 8.0, 140]
[150.0, 8.0, 140]
[150.0, 58.0, 140]
[200.0, 58.0, 140]
[[200, 58, 140], [200, 57.0, 140], [200, 56.0, 140], [200, 55.0, 140], [200, 54.0, 140], [200, 53.0, 140], [200, 52.0, 140], [200, 51.0, 140], [200, 50.0, 140], [200, 49.0, 140], [200, 48.0, 140], [200, 47.0, 140], [200, 46.0, 140], [200, 45.0, 140], [200, 44.0, 140], [200, 43.0, 140], [200, 42.0, 140], [200, 41.0, 140], [200, 40.0, 140], [200, 39.0, 140], [200, 38.0, 140], [200, 37.0, 140], [200, 36.0, 140], [200, 35.0, 140], [200, 34.0, 140], [200, 33.0, 140], [200, 32.0, 140], [200, 31.0, 140], [200, 30.0, 140], [200, 29.0, 140], [200, 28.0, 140], [200, 27.0, 140], [200, 26.0, 140], [200, 25.0, 140], [200, 24.0, 140], [200, 23.0, 140], [200, 22.0, 140], [200, 21.0, 140], [200, 20.0, 140], [200, 19.0, 140], [200, 18.0, 140], [200, 17.0, 140], [200, 16.0, 140], [200, 15.0, 140], [200, 14.0, 140], [200, 13.0, 140], [200, 12.0, 140], [200, 11.0, 140], [200, 10.0, 140], [200, 9.0, 140], [200, 8.0, 140], [199.0, 8.0, 

[[200, 58, 140],
 [200, 57.0, 140],
 [200, 56.0, 140],
 [200, 55.0, 140],
 [200, 54.0, 140],
 [200, 53.0, 140],
 [200, 52.0, 140],
 [200, 51.0, 140],
 [200, 50.0, 140],
 [200, 49.0, 140],
 [200, 48.0, 140],
 [200, 47.0, 140],
 [200, 46.0, 140],
 [200, 45.0, 140],
 [200, 44.0, 140],
 [200, 43.0, 140],
 [200, 42.0, 140],
 [200, 41.0, 140],
 [200, 40.0, 140],
 [200, 39.0, 140],
 [200, 38.0, 140],
 [200, 37.0, 140],
 [200, 36.0, 140],
 [200, 35.0, 140],
 [200, 34.0, 140],
 [200, 33.0, 140],
 [200, 32.0, 140],
 [200, 31.0, 140],
 [200, 30.0, 140],
 [200, 29.0, 140],
 [200, 28.0, 140],
 [200, 27.0, 140],
 [200, 26.0, 140],
 [200, 25.0, 140],
 [200, 24.0, 140],
 [200, 23.0, 140],
 [200, 22.0, 140],
 [200, 21.0, 140],
 [200, 20.0, 140],
 [200, 19.0, 140],
 [200, 18.0, 140],
 [200, 17.0, 140],
 [200, 16.0, 140],
 [200, 15.0, 140],
 [200, 14.0, 140],
 [200, 13.0, 140],
 [200, 12.0, 140],
 [200, 11.0, 140],
 [200, 10.0, 140],
 [200, 9.0, 140],
 [200, 8.0, 140],
 [199.0, 8.0, 140],
 [198.0, 8.0, 1

In [10]:
# ########轨迹运动之前可先调整一下 pid，以获得更好的曲线平滑度
ro.set_pid(joint_num=1, P=10, I=8, D=0.75)
ro.set_pid(joint_num=2, P=25, I=9, D=0.95)
ro.set_pid(joint_num=3, P=25, I=9, D=0.95)
ro.set_pid(joint_num=4, P=10, I=9, D=3.2)
ro.set_pid(joint_num=5, P=12, I=5, D=0.1)
ro.set_pid(joint_num=6, P=12, I=5, D=0.096)

位置增益 P：10.0
积分增益 I：8.0
转速增益 D：0.75
机械臂第  1  号关节的 PID 修改为： [10.0, 8.0, 0.75]
位置增益 P：25.0
积分增益 I：9.0
转速增益 D：0.949999988079071
机械臂第  2  号关节的 PID 修改为： [25.0, 9.0, 0.949999988079071]
位置增益 P：25.0
积分增益 I：9.0
转速增益 D：0.949999988079071
机械臂第  3  号关节的 PID 修改为： [25.0, 9.0, 0.949999988079071]
位置增益 P：10.0
积分增益 I：9.0
转速增益 D：3.200000047683716
机械臂第  4  号关节的 PID 修改为： [10.0, 9.0, 3.200000047683716]
位置增益 P：12.0
积分增益 I：5.0
转速增益 D：0.10000000149011612
机械臂第  5  号关节的 PID 修改为： [12.0, 5.0, 0.10000000149011612]
位置增益 P：12.0
积分增益 I：5.0
转速增益 D：0.09600000083446503
机械臂第  6  号关节的 PID 修改为： [12.0, 5.0, 0.09600000083446503]


True

In [11]:
# ########控制机械臂末端连续运动到多个指定位置和姿态函数(必须单独一次性使用)
# # ro.set_arm_poses(pls_temp=pl_list, theta_P_R_Ys_temp=[[0, 90, 0]], t=10) # 控制机械臂末端连续运动到多个指定位置和姿态函数(必须单独一次性使用)
ro.set_arm_poses_curve_pre(pls_temp=pl_list, theta_P_R_Ys_temp=[[0, 90, 0]]) # 预设机械臂末端轨迹函数
ro.set_arm_poses_curve_start_point(2) # 运动到轨迹起始位置函数
while True:
    ro.set_arm_poses_curve_do(5) # 末端轨迹执行函数，参数为大致运行时间

t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time
t >= n * bit_time


KeyboardInterrupt: 

读取机械臂当前坐标

In [9]:
# 初始和目标关节角
state0 = [0.0, 90.0, -30.0, 0.0, -0.0, -0.0]
state1 = [0.0, 90.0, -90.0, 0.0, -0.0, -0.0]

# 计算初始和目标位姿
T0 = DFbot.fkine(state0)
T1 = DFbot.fkine(state1)

In [8]:
T0

  -0.1366   -0.4268   -0.894    -0.114     
   0.2725    0.8515   -0.4481    0.2631    
   0.9524   -0.3048    0         0.06399   
   0         0         0         1         


In [10]:
# 生成笛卡尔空间轨迹（10个位姿）
cartesian_traj = ctraj(T0, T1, 10)

# 使用逆运动学将每个位姿转换为关节角
joint_traj = []
q_current = state0  # 初始猜测
for Ti in cartesian_traj:
    sol = DFbot.ikine_LM(Ti, q0=q_current)  # 直接调用 ikine_LM 方法
    if sol.success:
        joint_traj.append(sol.q)
        q_current = sol.q  # 更新当前解用于下一次迭代
    else:
        print("IK failed for a pose")

# 转换为 numpy 数组
joint_traj = np.array(joint_traj)

# 绘制动画
DFbot.plot(joint_traj, backend='pyplot', movie='panda3.gif')

PyPlot3D backend, t = 0.49999999999999994, scene:
  robot: Text(0.0, 0.0, 'DFbot')

In [43]:
joint_radian = np.array(joint_traj) * 180 / np.pi

joint_radian

array([[-8.62869888e+01,  1.39271665e+01, -3.42552054e+01,
         1.10328029e+02, -1.57092856e+02,  1.07746762e+02],
       [-8.73012806e+01,  2.10215150e+01, -4.79222738e+01,
         1.16900754e+02, -1.56078564e+02,  1.12253800e+02],
       [-9.09531902e+01,  3.59746177e+01, -7.59313584e+01,
         1.29956741e+02, -1.52426654e+02,  1.25774908e+02],
       [-1.00249915e+02,  5.50485978e+01, -1.09096141e+02,
         1.44048026e+02, -1.43129955e+02,  1.48310677e+02],
       [-1.24027797e+02,  7.81065334e+01, -1.41043358e+02,
         1.52936826e+02, -1.19352048e+02,  1.75352284e+02],
       [ 1.62144312e+02,  1.28964272e+02, -1.66683835e+02,
         1.27719592e+02, -4.55241513e+01, -1.57605548e+02],
       [ 1.16626759e+02,  1.13500343e+02, -1.55821071e+02,
         1.32723728e+02, -6.60229006e-03, -1.30966319e+02],
       [ 1.16619539e+02,  9.01000479e+01, -1.33759601e+02,
         1.30337069e+02,  6.16810046e-04, -1.04705636e+02],
       [ 1.16620155e+02,  7.73705563e+01, -1.162

In [45]:
ro.set_arm_joints(joint_radian)

角度参数有误！


False